# SiPadHits Energy and Position

Beginner version: use ROOT's `RDataFrame` to read leaves from the `events` tree, make a simple energy histogram, then select hits with `energy > 0.005` and compare `position.x` before and after the selection.

In [10]:
from pathlib import Path

import ROOT

ROOT.gStyle.SetOptStat(0)

input_file = Path(
    "/home/llr/ilc/valeria/work/siwecal_k4sim/mydata/TB2026-06/Simulation/Generated/"
    "output_beam_e-_54GeV_xy_-45_45_sigx13.75_sigy8.25_sigE0.02.edm4hep.root"
)

tree_name = "events"
energy_leaf = "SiPadHits.energy"
x_leaf = "SiPadHits.position.x"
energy_cut = 0.005

if not input_file.exists():
    raise FileNotFoundError(input_file)

input_file


PosixPath('/home/llr/ilc/valeria/work/siwecal_k4sim/mydata/TB2026-06/Simulation/Generated/output_beam_e-_54GeV_xy_-45_45_sigx13.75_sigy8.25_sigE0.02.edm4hep.root')

## Energy Histogram

`RDataFrame` opens the tree. `Histo1D` reads one leaf and fills a ROOT histogram.

In [14]:
df = ROOT.RDataFrame(tree_name, str(input_file))

h_energy = df.Histo1D(
    ("h_energy", "SiPadHits.energy;Energy [GeV];Hits", 100, 0.0, 0.06),
    energy_leaf,
)

print("number of hits:", int(h_energy.GetEntries()))
print("mean energy [GeV]:", h_energy.GetMean())

canvas_energy = ROOT.TCanvas("canvas_energy", "SiPadHits energy", 700, 500)
h_energy.Draw()
canvas_energy.Draw()


number of hits: 53451
mean energy [GeV]: 0.001092127346210975


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_energy


## Select Hits

The two short aliases make the selection easier to read: `energy` is `SiPadHits.energy`, and `x` is `SiPadHits.position.x`.

In [12]:
df_hits = df.Alias("energy", energy_leaf).Alias("x", x_leaf)
df_selected = df_hits.Define("selected_x", f"x[energy > {energy_cut}]")

h_x_all = df_hits.Histo1D(
    ("h_x_all", "position.x before selection;position.x [mm];Hits", 100, -100.0, 50.0),
    "x",
)
h_x_selected = df_selected.Histo1D(
    ("h_x_selected", "position.x after energy selection;position.x [mm];Hits", 100, -100.0, 50.0),
    "selected_x",
)

print("all hits:", int(h_x_all.GetEntries()))
print(f"hits with energy > {energy_cut}:", int(h_x_selected.GetEntries()))
print("mean x before selection [mm]:", h_x_all.GetMean())
print("mean x after selection [mm]:", h_x_selected.GetMean())


all hits: 53451
hits with energy > 0.005: 2018
mean x before selection [mm]: -42.32131617183761
mean x after selection [mm]: -45.04558131527026


In [13]:
canvas_x = ROOT.TCanvas("canvas_x", "position.x comparison", 700, 500)

h_x_all.SetLineColor(ROOT.kBlue + 1)
h_x_all.SetLineWidth(3)
h_x_selected.SetLineColor(ROOT.kRed + 1)
h_x_selected.SetLineWidth(3)

h_x_all_norm = h_x_all.DrawNormalized("hist")
h_x_selected_norm = h_x_selected.DrawNormalized("hist same")

legend = ROOT.TLegend(0.55, 0.72, 0.88, 0.88)
legend.AddEntry(h_x_all_norm, "all hits", "l")
legend.AddEntry(h_x_selected_norm, f"energy > {energy_cut}", "l")
legend.Draw()

canvas_x.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: canvas_x
